# 🛒 Amazon Reviews — Sentiment Analysis with RoBERTa

Uses `cardiffnlp/twitter-roberta-base-sentiment` — a RoBERTa model fine-tuned on 58M tweets.

**Output:** 3-class sentiment per review → **Negative / Neutral / Positive**

---

## 📦 Step 1: Install & Import Dependencies

In [ ]:
!pip install transformers torch scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
print('✅ Libraries imported!')
print(f'GPU available: {torch.cuda.is_available()}')

## 📥 Step 2: Load Dataset

In [ ]:
# ── Download from Kaggle ──────────────────────────────────────────
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d snap/amazon-fine-food-reviews

from zipfile import ZipFile
with ZipFile('amazon-fine-food-reviews.zip', 'r') as z:
    z.extractall()

# ── Load CSV ──────────────────────────────────────────────────────
df = pd.read_csv('Reviews.csv')
print(f'Full dataset shape: {df.shape}')

# Use 500 rows for speed — increase for production use
df = df.head(500).copy()
print(f'Working subset: {df.shape}')
df[['Id', 'Score', 'Summary', 'Text']].head()

## 🔍 Step 3: Exploratory Data Analysis

In [ ]:
print('Missing values:')
print(df[['Score', 'Text']].isnull().sum())
print(f'\nScore distribution:')
print(df['Score'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Star rating counts
ax = sns.countplot(data=df, x='Score', palette='viridis', ax=axes[0])
axes[0].set_title('Distribution of Star Ratings', fontsize=13)
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2, p.get_height() + 1),
                ha='center', fontsize=10)

# Review text length distribution
df['text_length'] = df['Text'].astype(str).apply(len)
axes[1].hist(df['text_length'], bins=30, color='steelblue', edgecolor='black')
axes[1].set_title('Review Text Length Distribution', fontsize=13)
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 🏷️ Step 4: Create Ground Truth Labels

Map star ratings → 3-class sentiment to evaluate model predictions against.

In [ ]:
def score_to_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['true_sentiment'] = df['Score'].apply(score_to_sentiment)
print('Ground truth label distribution:')
print(df['true_sentiment'].value_counts())

## 🤖 Step 5: Load RoBERTa Model

In [ ]:
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'

print(f'Loading model: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
model.eval()

# Label mapping for this model
LABELS = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print(f'✅ Model loaded on: {device}')

## 🔮 Step 6: Run Sentiment Inference

In [ ]:
def predict_roberta(text):
    """
    Returns a dict with:
      - predicted_label : 'Negative' / 'Neutral' / 'Positive'
      - neg_score, neu_score, pos_score : softmax probabilities
    """
    try:
        encoded = tokenizer(
            str(text),
            return_tensors='pt',
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        scores = softmax(output.logits[0].cpu().numpy())
        pred   = int(np.argmax(scores))

        return {
            'predicted_sentiment': LABELS[pred],
            'neg_score': round(float(scores[0]), 4),
            'neu_score': round(float(scores[1]), 4),
            'pos_score': round(float(scores[2]), 4)
        }
    except Exception as e:
        return {'predicted_sentiment': None,
                'neg_score': None, 'neu_score': None, 'pos_score': None}


# Run on all rows
print(f'Running RoBERTa on {len(df)} reviews... (may take a few minutes without GPU)')
results = df['Text'].apply(predict_roberta)
results_df = pd.DataFrame(results.tolist())

df = pd.concat([df.reset_index(drop=True), results_df], axis=1)
print('✅ Inference complete!')
df[['Score', 'true_sentiment', 'predicted_sentiment', 'neg_score', 'neu_score', 'pos_score']].head(10)

## 📊 Step 7: Evaluate the Model

In [ ]:
# Drop rows where inference failed
eval_df = df.dropna(subset=['predicted_sentiment']).copy()

acc = accuracy_score(eval_df['true_sentiment'], eval_df['predicted_sentiment'])
print(f'Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print('\nDetailed Classification Report:')
print(classification_report(
    eval_df['true_sentiment'],
    eval_df['predicted_sentiment'],
    target_names=['Negative', 'Neutral', 'Positive']
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(
    eval_df['true_sentiment'],
    eval_df['predicted_sentiment'],
    labels=['Negative', 'Neutral', 'Positive']
)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('RoBERTa — Confusion Matrix', fontsize=14)
plt.ylabel('Actual (Star Rating)')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Average confidence scores per star rating
score_group = eval_df.groupby('Score')[['neg_score', 'neu_score', 'pos_score']].mean()

score_group.plot(kind='bar', figsize=(10, 5), colormap='coolwarm', edgecolor='black')
plt.title('Average RoBERTa Scores by Star Rating', fontsize=13)
plt.xlabel('Star Rating')
plt.ylabel('Average Probability')
plt.xticks(rotation=0)
plt.legend(['Negative', 'Neutral', 'Positive'])
plt.tight_layout()
plt.show()

## 💾 Step 8: Save Results

In [ ]:
output_cols = ['Id', 'Score', 'Summary', 'Text',
               'true_sentiment', 'predicted_sentiment',
               'neg_score', 'neu_score', 'pos_score']

df[output_cols].to_csv('roberta_sentiment_results.csv', index=False)
print('✅ Results saved to roberta_sentiment_results.csv')

## 🔮 Step 9: Predict on Custom Text

In [ ]:
def predict_custom(text):
    result = predict_roberta(text)
    label  = result['predicted_sentiment']
    emoji  = {'Positive': '✅', 'Neutral': '😐', 'Negative': '❌'}.get(label, '?')
    print(f'Text     : "{text[:100]}"')
    print(f'Sentiment: {emoji} {label}')
    print(f'Scores   → Negative: {result["neg_score"]:.3f} | '
          f'Neutral: {result["neu_score"]:.3f} | '
          f'Positive: {result["pos_score"]:.3f}')
    print()

# ── Try your own reviews below! ───────────────────────────────────
predict_custom("Absolutely love this product! Will definitely buy again.")
predict_custom("Terrible quality. Broke after one use. Total waste of money.")
predict_custom("It's okay, nothing special but gets the job done.")